# 03 Lingkungan dan Reproduksibilitas


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Find paper root from the standard Drive path or from the local package.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next((p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()), None)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')
LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('03_Lingkungan_dan_Reproduksibilitas')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
METHODS = list(el.METHODS)
print('PAPER_ROOT:', PAPER_ROOT)
print('CARE config:', CARE_CONFIG)
print('CARE metadata:', el.frozen_care_metadata())


import platform
import sklearn
import scipy

DATASET_FILE = lms_data('categorical')
CARE_FILE = LIB_DIR / 'care_km.py'

care_hash = el.sha256_file(CARE_FILE)
dataset_hash = el.sha256_file(DATASET_FILE)

df, X, FEATURES = el.load_lms_dataset(PAPER_ROOT)
audit = el.dataset_audit(df, FEATURES)
el.save_csv(audit, artifact_path('00_dataset_audit.csv','03_Lingkungan_dan_Reproduksibilitas'))

protocol = {
    'paper_root': str(PAPER_ROOT),
    'dataset_file': str(DATASET_FILE),
    'dataset_sha256': dataset_hash,
    'care_km_sha256': care_hash,
    'care_implementation_version': getattr(ck, 'CARE_IMPLEMENTATION_VERSION', 'unknown'),
    'K': K,
    'max_iter': el.MAX_ITER,
    'si_sample_size': el.SI_SAMPLE_SIZE,
    'methods': METHODS,
    'stochastic_methods': sorted(el.STOCHASTIC_METHODS),
    'care_config': CARE_CONFIG,
    'care_metadata': el.frozen_care_metadata(),
    'scale_treated_as_nominal_categories': True,
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
    'scipy': scipy.__version__,
}
(artifact_path('00_protocol.json','03_Lingkungan_dan_Reproduksibilitas')).write_text(
    json.dumps(protocol, indent=2),
    encoding='utf-8'
)

display(audit)
print(json.dumps(protocol, indent=2))
print('STATUS: PASS')